# GV Exercise – Modular Pipeline Notebook

This notebook demonstrates the exact same data preparation, training, and inference flow as the production code by importing the reusable modules under `src/`.

It intentionally avoids ad-hoc pandas logic so experiments stay aligned with the CLI scripts and MLflow-tracked runs.

In [ ]:
%pip install seaborn
%pip install optuna
%pip install shap

Note: you may need to restart the kernel to use updated packages.


## Phase 1: Environment Setup

Bootstrap project imports and configuration. This sets up the shared imports, config, and logging so experiments match CLI runs.

In [ ]:
import sys
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import xgboost as xgb

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.common import load_config
from src.common.logging import init_logging
from src.models.io import load_model_bundle
from src.predict.inference import predict_batch
from src.models.training import train_ranker
from src.models.training_pipeline import build_training_dataset
from src.features.pipeline import FeaturePipeline

cfg = load_config()
init_logging(cfg.logging)
cfg


## Phase 2: Data Loading & Feature Engineering

Build the training dataset directly from packaged loaders. This pulls packaged training data and cleans it with the same helpers used in production. It also performs manual feature engineering and builds the feature matrix.

### 2.1 Data Loading & Cleaning

Load the raw training data and target labels from the configured data directories.

In [ ]:
from src.data.loaders import load_raw, clean, load_targets
raw_training = load_raw(cfg.data, dataset='training')
clean_training = clean(raw_training)
raw_ranking = load_raw(cfg.data, dataset='ranking')
clean_ranking = clean(raw_ranking)
target_df = load_targets(cfg.data)
print('Training:', { 'experience': clean_training.experience.shape, 'education': clean_training.education.shape, 'company_info': clean_training.company_info.shape })
print('Ranking:', { 'experience': clean_ranking.experience.shape, 'education': clean_ranking.education.shape, 'company_info': clean_ranking.company_info.shape })
print('Targets:', target_df.shape, list(target_df.columns))


### 2.5 Feature Matrix Construction

Build the feature matrix from the clean training data using the initialized pipeline.

In [ ]:
training_df = build_training_dataset(clean_training, clean_ranking, target_df, cfg.features)

### 2.7 Manual Feature Engineering

Apply any manual feature engineering steps, such as handling missing values or creating custom flags, and define the final list of feature columns.

In [ ]:
# Define feature columns from config
feature_columns = cfg.features.selected_columns

## Phase 3: Data Splitting

Perform per-industry cohort splitting with temporal ordering. We split the data by industry and use temporal ordering (older companies in train, newer in test) to simulate a realistic production scenario.

In [ ]:
from src.data.splitters import split_by_industry

# Split by industry with temporal ordering (older companies in train, newer in test)
X_train, y_train, X_test, y_test, train_df, test_df, train_groups, test_groups, industry_names = split_by_industry(
    df=training_df,
    feature_cols=feature_columns,
    target_col=cfg.features.target_column,
    train_ratio=0.8
)

print(f"\nTraining rows: {len(training_df)} | Feature columns: {len(feature_columns)}")
training_df.head()

## Phase 4: Data Validation

Quickly verify data integrity by checking for any records with the default '2025' year, which would indicate missing founded dates.

In [ ]:
(training_df['company_founded'] == 2025).sum()

## Phase 5: Model Training

Train the XGBoost ranker using config-driven hyperparameters. This ensures the training process in the notebook mirrors the production pipeline.

In [ ]:
params = cfg.model.hyperparameters.copy()

ranker, metrics = train_ranker(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    train_groups=train_groups,
    test_groups=test_groups,
    industry_names=industry_names,
    params=params,
    tracking_uri=cfg.tracking.uri,
    experiment_name=cfg.tracking.experiment_name,
    run_name=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)

## Phase 6: Model Persistence

Persist artifacts alongside metadata, identical to the CLI script. This saves the trained model and auxiliary metadata (like feature columns and outcome lookups) to the artifacts directory.

In [ ]:
# Persist the trained bundle locally to mirror CLI artifact structure
run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
# Organize notebook runs under timestamped folders for traceability
run_dir = cfg.model.artifact_dir / "notebook_runs" / run_name
run_dir.mkdir(parents=True, exist_ok=True)

model_path = run_dir / "ranker.json"
artifacts_path = run_dir / "artifacts.json"
# Save the booster for reuse in inference cells
ranker.save_model(model_path)

# Bundle auxiliary metadata used during inference
artifacts = {
    "feature_columns": feature_columns,
}
# Persist metadata next to the model bundle
artifacts_path.write_text(json.dumps(artifacts, indent=2))

run_dir


## Phase 7: Inference

Load the freshly written bundle and score current founders. This demonstrates how to reload the saved model and use it to generate predictions for new data.

**Note on SHAP Explanations:**
We enable SHAP value calculation (`shap_top_n=3`) to provide local interpretability for each prediction. The `explanation` column in the output dataframe lists the top 3 features contributing to the score for each founder.

In [ ]:
# Reload the saved bundle and score the latest founders
bundle = load_model_bundle(
    registry_cfg=cfg.registry,
    project_root=cfg.project.root,
    model_root=run_dir,
)

# Pull current founders and clean them the same way as training data
raw_ranking = load_raw(cfg.data, dataset="ranking")
clean_ranking = clean(raw_ranking)

# Build the feature matrix for ranking founders using the same feature config
feature_pipeline = FeaturePipeline(cfg.features)
ranking_features = feature_pipeline.build_matrix(clean_ranking)

# Run batch prediction with SHAP explanations enabled (top 3 features)
predictions = predict_batch(bundle, ranking_features, shap_top_n=3)
predictions.head()


## Phase 8: Ranking Explanation

Visualize the feature importance and impact using SHAP (SHapley Additive exPlanations). This provides a global view of what drives the model's ranking decisions.

In [ ]:
import shap
import matplotlib.pyplot as plt

# Use the estimator from the loaded bundle
model = bundle.estimator

# Select only the feature columns for SHAP calculation
X_ranking = ranking_features.select_columns(bundle.feature_columns)

# Initialize the explainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_ranking)

# Handle case where shap_values might be a list (for some XGBoost objectives)
if isinstance(shap_values, list):
    shap_values = shap_values[0]

print("SHAP Summary Plot (Global Feature Importance):")
shap.summary_plot(shap_values, X_ranking, plot_type="bar")
plt.show()

print("SHAP Beeswarm Plot (Feature Impact Direction):")
shap.summary_plot(shap_values, X_ranking)
plt.show()

# Per-industry SHAP summary (excluding unknown)
industry_series = ranking_features.frame["industry"]
valid_industries = [
    value
    for value in sorted(industry_series.dropna().unique())
    if isinstance(value, str) and value and value.lower() != "unknown"
]

for industry in valid_industries:
    mask = industry_series == industry
    print(f"\nSHAP summary for industry: {industry} (n={mask.sum()})")
    shap.summary_plot(
        shap_values[mask.values],
        X_ranking[mask.values],
        show=False,
    )
    plt.title(f"SHAP summary – {industry}")
    plt.show()


> **Why does `artifacts.json` store company outcomes?**
>
> The mapping is used to label past companies when computing founder experience features (e.g., counting prior 10x exits). Each founder example still receives an individual label during training (`relevance_grade`) and we score founders after training, but we need company-level multiples to quantify their historical track records.

In [ ]:
predictions